# Reconstructing the Cursor confidence project
## From raw coding answers to a trained confidence model

**Reconstruction created October 14, 2026.** This is a new, executable teaching reconstruction using public benchmark data and the current local implementation. It is not a recovered copy of the unavailable original work, private Cursor data, or evidence that the original work used Jupyter.

The question is: **Can a small model estimate whether an AI-generated coding answer will pass its benchmark tests, and do calibration adjustments make those probabilities more useful?**

You will see the actual raw record, all ten answers to one problem, all sixteen numeric features, the training matrix, learned weights, probability calculations, calibration updates, and held-out evaluation.

This notebook refits models on an already examined dataset. Its results are a reproducibility exercise, not a new untouched confirmatory experiment. No generated program is executed. Published pass/fail outcomes are used as labels.

### How to read it
Read the explanation immediately above each cell, then inspect its output. The notebook is saved with outputs so it can also be read without rerunning. Rerunning uses the companion Python files in this repository. On a fresh clone, the first cell downloads approximately 450 MiB of pinned public data into `research/data/`. Generated reports stay in `reconstruction/`.

### Run from a fresh clone

Use Python 3.11 or later. From the repository root:

```bash
python -m pip install -r requirements.txt
jupyter lab reconstruction/cursor-confidence-walkthrough.ipynb
```

Saved outputs can be read directly on GitHub without downloading the dataset. The original source data is [CALIBRI](https://huggingface.co/datasets/lavis-nlp/CALIBRI); its dataset and benchmark terms continue to apply.


## 1. Load the tools and locate the public data

The project folder contains the reusable extractor and calibration algorithm. The notebook makes the important fitting steps explicit rather than hiding the entire experiment behind a single function.

Python, NumPy, pandas, PyArrow, SciPy, scikit-learn, and Jupyter are the tools for this reconstruction. The environment name does not change the statistical method.


In [1]:
from pathlib import Path
from html import escape
import hashlib
import importlib.metadata
import inspect
import json
import sys
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import Code, HTML, Markdown, display
from scipy.special import expit, logit
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

PROJECT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "research" / "data.py").exists()
)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from research.data import FEATURES, features, digest, prepare
from research.methods import (
    audit, budget_threshold, clustered_interval, export_classifier,
    fit_patches, group_thresholds, membership, policy,
    predict_classifier, replay, route_mask,
)
from research.run import partition_tasks

RESEARCH = PROJECT / "research"
OUTPUT = PROJECT / "reconstruction"
OUTPUT.mkdir(exist_ok=True)
REFERENCE_RUN = RESEARCH / "runs" / "a8630092fa9e"
protocol = json.loads((REFERENCE_RUN / "protocol.json").read_text())
prepared_path = RESEARCH / "data" / "prepared.json"
if not prepared_path.exists():
    prepare(protocol)
prepared = json.loads(prepared_path.read_text())
assert digest(RESEARCH / "data" / "prepared.json") == (RESEARCH / "data" / "prepared.sha256").read_text().strip()
assert prepared["extractor_hash"] == digest(RESEARCH / "data.py")
assert prepared["revision"] == protocol["revision"]

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")

display(pd.DataFrame({
    "package": ["numpy", "scipy", "scikit-learn", "pandas", "pyarrow"],
    "version": [importlib.metadata.version(name) for name in ["numpy", "scipy", "scikit-learn", "pandas", "pyarrow"]],
}))
print("Public dataset:", protocol["dataset"])
print("Pinned revision:", protocol["revision"])
print("Generated output directory:", OUTPUT.relative_to(PROJECT))


,package,version
0,numpy,2.1.3
1,scipy,1.15.3
2,scikit-learn,1.6.1
3,pandas,2.2.3
4,pyarrow,24.0.0


Public dataset: lavis-nlp/CALIBRI
Pinned revision: 7a4a7dc7aceb1b4a426c48761d0f128f2daf6a54
Generated output directory: reconstruction


## 2. What the raw dataset actually looks like

The source files are Parquet tables, not a spreadsheet someone manually typed.

**One raw source row is one coding problem, containing ten generated answers.** The prompt is shared; program, output, correctness, token records, and code-token indices are length-ten lists.

- **prompt:** the coding problem.
- **program:** extracted executable-looking code for each attempt.
- **output:** the fuller generated response, potentially including explanation.
- **is_correct:** the publisher's pass/fail result for each attempt.
- **token_logprobs:** recorded token strings and log probabilities for each response.
- **code_token_idx:** supplied code-token locations; the current extractor does not use them.
- **id, name, difficulty, language, model:** metadata. Difficulty and task ID do not enter the confidence model.

We use a real held-out problem, **Yarik and Array**, to make the walkthrough concrete. It was selected for explanation, not used to tune the trained model.


In [2]:
EXAMPLE_ID = "1899_C"
CANDIDATE_INDEX = 4
source_path = RESEARCH / "data" / protocol["primary"] / "test.parquet"
source_info = next(item for item in prepared["sources"] if item["config"] == protocol["primary"] and item["split"] == "test")
assert digest(source_path) == source_info["sha256"]
parquet = pq.ParquetFile(source_path)

raw = next(
    row
    for batch in parquet.iter_batches(batch_size=8)
    for row in batch.to_pylist()
    if row["id"] == EXAMPLE_ID
)
example_task = next(task for task in prepared["tasks"] if task["id"] == EXAMPLE_ID)

field_rows = []
for name, value in raw.items():
    if isinstance(value, list):
        size = f"{len(value)} attempts"
    elif isinstance(value, str):
        size = f"{len(value):,} characters"
    else:
        size = ""
    field_rows.append({"field": name, "Python type": type(value).__name__, "size": size})
display(pd.DataFrame(field_rows))
print("Task:", raw["id"], raw["name"])
print("Source model:", raw["model"])
print("Raw source rows in this test file:", parquet.metadata.num_rows)
print("Recorded labels for attempts 1–10:", raw["is_correct"])

raw_export = OUTPUT / "raw-example-1899-c.json"
raw_export.write_text(json.dumps(raw, indent=2, ensure_ascii=False) + "\n")
print("Full unabridged example exported to:", raw_export.name)


,field,Python type,size
0,id,str,6 characters
1,name,str,18 characters
2,prompt,str,"1,251 characters"
3,program,list,10 attempts
4,output,list,10 attempts
5,language,str,6 characters
6,is_correct,list,10 attempts
7,token_logprobs,list,10 attempts
8,code_token_idx,list,10 attempts
9,difficulty,str,4 characters


Task: 1899_C C. Yarik and Array
Source model: qwen3-coder
Raw source rows in this test file: 264
Recorded labels for attempts 1–10: [True, True, True, True, False, True, False, True, True, False]
Full unabridged example exported to: raw-example-1899-c.json


In [3]:
raw_data = pd.read_parquet(
    RESEARCH / "data/livecodebench_qwen3/test.parquet"
)
raw_data.head()

,id,name,prompt,program,output,language,is_correct,token_logprobs,code_token_idx,difficulty,model
0,1899_C,C. Yarik and Array,A subarray is a continuous part of array.\n\nYarik recently found ...,[t = int(input())\nfor _ in range(t):\n n = int(input())\n a...,[I'll solve this step by step.\n\n## Problem Understanding\n\n1. W...,python,"[True, True, True, True, False, True, False, True, True, False]","[[[-0.005979032255709171, I], [-3.576278118089249E-7, 'll], [0.0, ...","[[1143, 1305], [993, 1213], [989, 1209], [1191, 1343], [825, 1009]...",hard,qwen3-coder
1,3387,minimum-operations-to-make-median-of-array-equal-to-k,You are given an integer array nums and a non-negative integer k. ...,"[class Solution:\n def minOperationsToMakeMedianK(self, nums: L...",[```python\nclass Solution:\n def minOperationsToMakeMedianK(se...,python,"[False, False, False, False, False, False, True, True, False, True]","[[[-0.4893374741077423, Looking], [0.0, python], [0.0, Ċ], [0.0, c...","[[0, 0], [1806, 1983], [0, 0], [940, 1103], [0, 0], [0, 0], [850, ...",medium,qwen3-coder
2,abc369_c,Count Arithmetic Subarrays,"You are given a sequence of N positive integers A=(A_1,A_2,\dots,A...","[n = int(input())\na = list(map(int, input().split()))\n\ncount = ...",[I'll solve this step by step.\n\n## Problem Understanding\n\nWe n...,python,"[False, False, False, False, False, False, False, False, False, Fa...","[[[-0.0817812830209732, I], [-0.000015020257706055418, 'll], [-1.1...","[[1201, 1335], [1427, 1570], [951, 1128], [1684, 1811], [698, 850]...",medium,qwen3-coder
3,3408,count-the-number-of-special-characters-i,You are given a string word. A letter is called special if it appe...,"[class Solution:\n def numberOfSpecialChars(self, word: str) ->...","[```python\nclass Solution:\n def numberOfSpecialChars(self, wo...",python,"[True, True, True, True, True, True, True, True, True, True]","[[[-0.1217416301369667, ```], [0.0, python], [0.0, Ċ], [0.0, class...","[[0, 130], [0, 130], [0, 130], [0, 130], [0, 130], [0, 130], [0, 1...",easy,qwen3-coder
4,abc324_f,Beautiful Path,There is a directed graph with N vertices and M edges. Each edge h...,"[, import sys\nfrom collections import deque\n\ndef can_achieve_ra...",[I'll solve this step by step.\n\n## Problem Understanding\n\nWe n...,python,"[False, False, False, False, False, False, False, False, False, Fa...","[[[-0.18737560510635376, I], [-0.000008940656698541716, 'll], [-1....","[[0, 0], [854, 1277], [0, 0], [0, 0], [1034, 1555], [0, 0], [0, 0]...",hard,qwen3-coder


In [4]:
raw_data["prompt"][1]

'You are given an integer array nums and a non-negative integer k. In one operation, you can increase or decrease any element by 1.\nReturn the minimum number of operations needed to make the median of nums equal to k.\nThe median of an array is defined as the middle element of the array when it is sorted in non-decreasing order. If there are two choices for a median, the larger of the two values is taken.\n \nExample 1:\n\nInput: nums = [2,5,6,8,5], k = 4\nOutput: 2\nExplanation:\nWe can subtract one from nums[1] and nums[4] to obtain [2, 4, 6, 8, 4]. The median of the resulting array is equal to k.\n\nExample 2:\n\nInput: nums = [2,5,6,8,5], k = 7\nOutput: 3\nExplanation:\nWe can add one to nums[1] twice and add one to nums[2] once to obtain [2, 7, 7, 8, 5].\n\nExample 3:\n\nInput: nums = [1,2,3,4,5,6], k = 4\nOutput: 0\nExplanation:\nThe median of the array is already equal to k.\n\n \nConstraints:\n\n1 <= nums.length <= 2 * 10^5\n1 <= nums[i] <= 10^9\n1 <= k <= 10^9'

### The actual coding question

This is the full stored prompt, including constraints and examples. The predictor will not read it like an LLM; it receives only the numerical measurements extracted later.


In [5]:
print(raw["prompt"])


A subarray is a continuous part of array.

Yarik recently found an array $a$ of $n$ elements and became very interested in finding the maximum sum of a non empty subarray. However, Yarik doesn't like consecutive integers with the same parity, so the subarray he chooses must have alternating parities for adjacent elements.

For example, $[1, 2, 3]$ is acceptable, but $[1, 2, 4]$ is not, as $2$ and $4$ are both even and adjacent.

You need to help Yarik by finding the maximum sum of such a subarray.

Input

The first line contains an integer $t$ $(1 \le t \le 10^4)$ — number of test cases. Each test case is described as follows.

The first line of each test case contains an integer $n$ $(1 \le n \le 2 \cdot 10^5)$ — length of the array.

The second line of each test case contains $n$ integers $a_1, a_2, \dots, a_n$ $(-10^3 \le a_i \le 10^3)$ — elements of the array.

It is guaranteed that the sum of $n$ for all test cases does not exceed $2 \cdot 10^5$.

Output

For each test case, outpu

### All ten generated programs for that one question

Each expandable section contains an entire extracted program and its publisher label. Attempt numbers here start at 1; array indices in the code start at 0.

The dataset preserves empty and invalid answers rather than quietly removing easy failures. Passing the publisher's benchmark tests does not establish correctness for every possible input.


In [6]:
sections = []
for index, (program, label) in enumerate(zip(raw["program"], raw["is_correct"])):
    status = "PASS" if label else "FAIL"
    sections.append(
        f'<details><summary>Attempt {index + 1} · {status} · {len(program):,} code characters</summary>'
        f'<pre>{escape(program) if program else "[empty extracted program]"}</pre></details>'
    )
display(HTML("\n".join(sections)))
display(pd.DataFrame([
    {
        "attempt": index + 1,
        "code characters": len(program),
        "code lines": len(program.splitlines()),
        "recorded output tokens": len(raw["token_logprobs"][index]),
        "label y": int(raw["is_correct"][index]),
    }
    for index, program in enumerate(raw["program"])
]))


,attempt,code characters,code lines,recorded output tokens,label y
0,1,599,20,1307,1
1,2,777,29,1215,1
2,3,777,29,1211,1
3,4,571,22,1345,1
4,5,767,26,1011,0
5,6,755,29,1076,1
6,7,788,30,873,0
7,8,613,25,1225,1
8,9,798,29,573,1
9,10,804,32,1150,0


### Follow attempt 5 closely

This answer has valid Python syntax, but its published label is failure. Syntax and confident wording do not establish correctness.

By inspection, when adjacent elements have different parity, this code always extends the previous sum. It does not consider restarting after a harmful negative prefix. For the array [-3, 4], that logic carries forward a sum of 1 rather than selecting the subarray [4]. This is an explanatory counterexample, not a claim that we reran the publisher's tests.


In [7]:
selected_program = raw["program"][CANDIDATE_INDEX]
selected_tokens = raw["token_logprobs"][CANDIDATE_INDEX]
display(Code(selected_program, language="python"))
print("Publisher's recorded label:", int(raw["is_correct"][CANDIDATE_INDEX]))
print("Full-response characters:", len(raw["output"][CANDIDATE_INDEX]))
print("Extracted-code characters:", len(selected_program))


t = int(input())
for _ in range(t):
    n = int(input())
    a = list(map(int, input().split()))
    
    max_sum = float('-inf')
    current_sum = 0
    
    for i in range(n):
        # Check if we can extend the current subarray
        if i == 0:
            # First element, always start with it
            current_sum = a[i]
        else:
            # Check if current element has different parity from previous
            if a[i] % 2 != a[i-1] % 2:
                # Different parity, we can extend
                current_sum += a[i]
            else:
                # Same parity, start fresh from current element
                current_sum = a[i]
        
        # Update maximum sum
        max_sum = max(max_sum, current_sum)
    
    print(max_sum)

Publisher's recorded label: 0
Full-response characters: 4216
Extracted-code characters: 767


## 3. Token log probabilities are raw measurements, not correctness probabilities

A token is a piece of generated text. Each pair in this source is stored as [log probability, token string]. A token log probability of -0.1 corresponds to a token probability of approximately 0.905.

The tokenizer may display markers such as Ġ for a space or Ċ for a newline. These are token spellings, not extra code characters.

**Important:** this implementation summarizes token probabilities across the full generated response, including any explanation. It does not restrict them to the extracted code. The first several tokens below make that visible.


In [8]:
display(pd.DataFrame([
    {
        "position": index,
        "stored token": pair[1],
        "log probability": float(pair[0]),
        "token probability exp(log p)": float(np.exp(float(pair[0]))),
    }
    for index, pair in enumerate(selected_tokens[:24])
]))
print("Number of recorded tokens for this answer:", len(selected_tokens))
print("Actual raw pairs, first six:")
print(json.dumps(selected_tokens[:6], indent=2, ensure_ascii=False))


,position,stored token,log probability,token probability exp(log p)
0,0,I,-0.007238,0.992788
1,1,'ll,-0.000000,1.000000
2,2,Ġsolve,0.000000,1.000000
3,3,Ġthis,0.000000,1.000000
4,4,Ġstep,-0.000000,1.000000
5,5,Ġby,0.000000,1.000000
6,6,Ġstep,-0.000001,0.999999
7,7,.ĊĊ,0.000000,1.000000
8,8,##,-0.000010,0.999990
9,9,ĠProblem,-0.000000,1.000000


Number of recorded tokens for this answer: 1011
Actual raw pairs, first six:
[
  [
    "-0.00723764393478632",
    "I"
  ],
  [
    "-3.576278118089249E-7",
    "'ll"
  ],
  [
    "0.0",
    "Ġsolve"
  ],
  [
    "0.0",
    "Ġthis"
  ],
  [
    "-3.576278118089249E-7",
    "Ġstep"
  ],
  [
    "0.0",
    "Ġby"
  ]
]


## 4. Turn a raw answer into exactly sixteen numbers

This step is **feature extraction**, not model training. The extractor measures the prompt, code, and token records. It does not inspect the pass/fail label, run the generated code, or read other attempts' outcomes.

For example, 767 code characters become ln(1 + 767) = 6.643790. Log transforms compress large counts. The later scaler is a separate operation.

AST depth is the depth of Python's entire syntax tree, not simply the number of nested loops. The table below shows every feature for the selected answer.


In [9]:
feature_explanations = {
    "log_prompt_chars": "ln(1 + number of prompt characters)",
    "log_code_chars": "ln(1 + number of extracted-code characters)",
    "log_code_lines": "ln(1 + number of extracted-code lines)",
    "log_output_tokens": "ln(1 + number of recorded output-token scores)",
    "mean_logprob": "Average recorded token log probability",
    "p10_logprob": "10th percentile of token log probabilities",
    "logprob_sd": "Standard deviation of token log probabilities",
    "low_probability_fraction": "Fraction of tokens whose log probability is below -2",
    "ast_depth": "Maximum Python syntax-tree depth, root at depth zero",
    "branches": "Count of if, conditional-expression, and match nodes",
    "loops": "Count of loops and comprehension generators",
    "functions": "Count of regular and async function definitions",
    "imports": "Count of import statements",
    "syntax_valid": "1 if nonempty code parses; this is not correctness",
    "empty_code": "1 if extracted code is empty or whitespace",
    "missing_logprobs": "1 if token-score records are unavailable",
}
selected_x = np.array(features(raw["prompt"], selected_program, selected_tokens))
np.testing.assert_allclose(selected_x, example_task["candidates"][CANDIDATE_INDEX]["features"])
display(pd.DataFrame({
    "feature": FEATURES,
    "extracted value": selected_x,
    "meaning": [feature_explanations[name] for name in FEATURES],
}))
print("Input vector shape:", selected_x.shape)
print("Target y is stored separately:", int(raw["is_correct"][CANDIDATE_INDEX]))


,feature,extracted value,meaning
0,log_prompt_chars,7.132498,ln(1 + number of prompt characters)
1,log_code_chars,6.643790,ln(1 + number of extracted-code characters)
2,log_code_lines,3.295837,ln(1 + number of extracted-code lines)
3,log_output_tokens,6.919684,ln(1 + number of recorded output-token scores)
4,mean_logprob,-0.167347,Average recorded token log probability
5,p10_logprob,-0.635513,10th percentile of token log probabilities
6,logprob_sd,0.298148,Standard deviation of token log probabilities
7,low_probability_fraction,0.000000,Fraction of tokens whose log probability is below -2
8,ast_depth,10.000000,"Maximum Python syntax-tree depth, root at depth zero"
9,branches,2.000000,"Count of if, conditional-expression, and match nodes"


Input vector shape: (16,)
Target y is stored separately: 0


### The extractor itself

This is the actual function used in the local reconstruction, shown so the transformations are inspectable. Parsing the code into a syntax tree is not executing it.


In [10]:
display(Code(inspect.getsource(features), language="python"))


def features(prompt: str, code: str, token_pairs: list) -> list[float]:
    # Parse the program as data only. Generated code is never executed.
    nodes, depth, valid = [], 0, 0
    if code.strip():
        try:
            tree = ast.parse(code)
            valid = 1
            stack = [(tree, 0)]
            while stack:
                node, level = stack.pop()
                nodes.append(node)
                depth = max(depth, level)
                stack.extend((child, level + 1) for child in ast.iter_child_nodes(node))
        except (SyntaxError, ValueError, RecursionError):
            pass
    probabilities = np.array([float(pair[0]) for pair in token_pairs], dtype=float)
    if not np.all(np.isfinite(probabilities)) or np.any(probabilities > 1e-6):
        raise ValueError("Invalid sampled-token log probabilities")
    missing = len(probabilities) == 0
    values = probabilities if not missing else np.array([0.0])
    return [
        math.log1p(len(prompt)), math.log1p(len(code)), math.log1p(len(code.splitlines())),
        math.log1p(len(probabilities)), float(values.mean()), float(np.quantile(values, .1)),
        float(values.std()), float(np.mean(values < -2)), float(depth),
        float(sum(isinstance(n, (ast.If, ast.IfExp, ast.Match)) for n in nodes)),
        float(sum(isinstance(n, (ast.For, ast.While, ast.AsyncFor, ast.comprehension)) for n in nodes)),
        float(sum(isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) for n in nodes)),
        float(sum(isinstance(n, (ast.Import, ast.ImportFrom)) for n in nodes)),
        float(valid), float(not code.strip()), float(missing),
    ]

## 5. Flatten task records into candidate examples

There are 1,055 tasks and ten primary answers per task, so there are 10,550 candidate examples.

For each candidate:
- **X** contains the sixteen measured inputs.
- **y** is 1 for a published pass and 0 for a published failure.
- **task_id** keeps related examples together. It is not a predictive feature.
- The alternate answer's outcome is reserved for the later routing simulation.

The preview below shows real rows, not invented teaching data. Metadata and y appear next to X for inspection, but they are not supplied as input columns.


In [11]:
flat_rows = []
for task in prepared["tasks"]:
    for candidate in task["candidates"]:
        flat_rows.append({
            "task_id": task["id"],
            "attempt": candidate["index"] + 1,
            "source_split": task["source_split"],
            **dict(zip(FEATURES, candidate["features"])),
            "y": candidate["label"],
        })
frame = pd.DataFrame(flat_rows)
print(f"Tasks: {len(prepared['tasks']):,}")
print(f"Candidate rows: {len(frame):,}")
print("Feature matrix shape:", frame[FEATURES].shape)
print("Target shape:", frame["y"].shape)
display(frame.head(20))
display(frame.loc[frame["task_id"] == EXAMPLE_ID])


Tasks: 1,055
Candidate rows: 10,550
Feature matrix shape: (10550, 16)
Target shape: (10550,)


,task_id,attempt,source_split,log_prompt_chars,log_code_chars,log_code_lines,log_output_tokens,mean_logprob,p10_logprob,logprob_sd,low_probability_fraction,ast_depth,branches,loops,functions,imports,syntax_valid,empty_code,missing_logprobs,y
0,1899_A,1,train,6.712956,0.000000,0.000000,7.601402,-0.210393,-0.829116,0.391028,0.001000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0
1,1899_A,2,train,6.712956,4.890349,2.079442,7.183112,-0.105691,-0.437510,0.250018,0.000000,6.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1
2,1899_A,3,train,6.712956,4.890349,2.079442,7.402452,-0.200692,-0.751819,0.369448,0.000610,6.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1
3,1899_A,4,train,6.712956,4.890349,2.079442,7.537963,-0.205239,-0.758773,0.366417,0.000000,6.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1
4,1899_A,5,train,6.712956,4.890349,2.079442,6.965080,-0.102853,-0.382763,0.258181,0.000000,6.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1
5,1899_A,6,train,6.712956,4.890349,2.079442,7.393878,-0.137154,-0.535583,0.295884,0.000000,6.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1
6,1899_A,7,train,6.712956,0.000000,0.000000,7.601402,-0.211286,-0.793026,0.385297,0.001500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0
7,1899_A,8,train,6.712956,4.890349,2.079442,6.782192,-0.069184,-0.252159,0.196969,0.000000,6.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1
8,1899_A,9,train,6.712956,4.890349,2.079442,7.449498,-0.147742,-0.532086,0.298122,0.000000,6.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1
9,1899_A,10,train,6.712956,4.890349,2.079442,7.497762,-0.141448,-0.531297,0.304299,0.000000,6.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1


,task_id,attempt,source_split,log_prompt_chars,log_code_chars,log_code_lines,log_output_tokens,mean_logprob,p10_logprob,logprob_sd,low_probability_fraction,ast_depth,branches,loops,functions,imports,syntax_valid,empty_code,missing_logprobs,y
7910,1899_C,1,test,7.132498,6.396930,3.044522,7.176255,-0.092165,-0.368395,0.233717,0.000765,10.000000,1.000000,2.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1
7911,1899_C,2,test,7.132498,6.656727,3.401197,7.103322,-0.027713,-0.023357,0.114316,0.000000,9.000000,2.000000,2.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1
7912,1899_C,3,test,7.132498,6.656727,3.401197,7.100027,-0.026949,-0.021663,0.112985,0.000000,9.000000,2.000000,2.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1
7913,1899_C,4,test,7.132498,6.349139,3.135494,7.204893,-0.098452,-0.394331,0.232320,0.000000,10.000000,2.000000,2.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1
7914,1899_C,5,test,7.132498,6.643790,3.295837,6.919684,-0.167347,-0.635513,0.298148,0.000000,10.000000,2.000000,2.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0
7915,1899_C,6,test,7.132498,6.628041,3.401197,6.981935,-0.027054,-0.023272,0.106995,0.000000,10.000000,2.000000,2.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1
7916,1899_C,7,test,7.132498,6.670766,3.433987,6.773080,-0.101509,-0.432159,0.216227,0.000000,10.000000,3.000000,2.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0
7917,1899_C,8,test,7.132498,6.419995,3.258097,7.111512,-0.049966,-0.131159,0.155488,0.000000,10.000000,2.000000,2.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1
7918,1899_C,9,test,7.132498,6.683361,3.401197,6.352629,-0.048990,-0.161421,0.134574,0.000000,9.000000,2.000000,2.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1
7919,1899_C,10,test,7.132498,6.690842,3.496508,7.048386,-0.028879,-0.029751,0.108891,0.000000,10.000000,3.000000,2.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0


## 6. Split by problem, not by generated answer

If attempt 1 for a problem were in training and attempt 2 in evaluation, the evaluation could benefit from having seen essentially the same question. All ten attempts therefore stay together.

The public validation and test task sets remain intact. The official training tasks are deterministically divided into:
- **Training:** fit feature scaling, logistic weights, and subgroup thresholds.
- **Calibration:** fit Platt scaling and subgroup corrections.
- **Validation:** select correction settings and routing cutoffs.
- **Test:** evaluate the frozen pipeline.

The existing test results have already been inspected historically. We preserve separation in the reconstruction, but do not call this rerun fresh confirmatory evidence.


In [12]:
parts, manifests = partition_tasks(prepared["tasks"], protocol)
group_names = protocol["groups"]
thresholds = group_thresholds(parts["train"]["x"])
for part in parts.values():
    part["groups"] = membership(part["x"], thresholds, group_names)

display(pd.DataFrame([
    {
        "partition": name,
        "distinct tasks": len(manifests[name]),
        "candidate rows": len(part["y"]),
        "X shape": str(part["x"].shape),
        "y shape": str(part["y"].shape),
    }
    for name, part in parts.items()
]))
for first, first_ids in manifests.items():
    for second, second_ids in manifests.items():
        if first != second:
            assert not set(first_ids).intersection(second_ids)
assert all(sum(row["id"] == task_id for row in part["rows"]) == 10
           for part in parts.values() for task_id in set(part["ids"]))
assert EXAMPLE_ID in manifests["test"]
print("Verified: no task crosses partitions; all sibling answers stay together.")


,partition,distinct tasks,candidate rows,X shape,y shape
0,train,316,3160,"(3160, 16)","(3160,)"
1,calibration,211,2110,"(2110, 16)","(2110,)"
2,validation,264,2640,"(2640, 16)","(2640,)"
3,test,264,2640,"(2640, 16)","(2640,)"


Verified: no task crosses partitions; all sibling answers stay together.


## 7. Train the confidence model

This is the central machine-learning step. We are not training the coding LLM. We fit a small classifier to its recorded outputs.

The scaler learns each feature's training mean and standard deviation. Logistic regression then learns sixteen coefficients and an intercept by minimizing penalized log loss. The L2 penalty discourages extreme weights.

There is no known 'true 80%' target for an individual row. The labels are only 0 and 1; probabilities are learned from patterns across many examples.

The .fit call below is real: rerunning this notebook refits the model on the 3,160 training examples.


In [13]:
base_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=0),
)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    with threadpool_limits(limits=1):
        base_model.fit(parts["train"]["x"], parts["train"]["y"])

base_scores = {
    name: base_model.predict_proba(part["x"])[:, 1]
    for name, part in parts.items()
}
base_artifact = export_classifier(base_model, FEATURES)
display(pd.DataFrame({
    "feature": FEATURES,
    "training mean": base_artifact["mean"],
    "training standard deviation": base_artifact["scale"],
    "learned coefficient beta": base_artifact["coefficients"],
}))
print("Learned intercept:", base_artifact["intercept"])
print("Optimizer iterations:", base_model[-1].n_iter_.tolist())
print("Constant columns receive scaler scale 1 to avoid division by zero.")


,feature,training mean,training standard deviation,learned coefficient beta
0,log_prompt_chars,7.179544,0.379332,-0.175905
1,log_code_chars,4.932393,2.919907,-0.517597
2,log_code_lines,2.399197,1.499139,-2.091595
3,log_output_tokens,6.585304,0.924368,-0.264206
4,mean_logprob,-0.119864,0.116076,-0.239882
5,p10_logprob,-0.383680,0.376589,-0.447582
6,logprob_sd,0.223839,0.134913,-1.672956
7,low_probability_fraction,0.000609,0.001629,-0.631655
8,ast_depth,6.730063,4.634240,-0.211947
9,branches,1.778165,2.663959,0.048774


Learned intercept: -1.7492070386359286
Optimizer iterations: [34]
Constant columns receive scaler scale 1 to avoid division by zero.


## 8. Calculate one prediction all the way through

For a new answer, apply the stored training scaling, multiply each standardized feature by its learned coefficient, and add the intercept. This gives a raw score, often called a logit. The sigmoid converts it to a number between 0 and 1.

The table shows every contribution for attempt 5. These are contributions to this fitted model's score, **not causal effects**. A negative coefficient does not prove that changing that feature would improve the code.

The test label is not used anywhere in this calculation.


In [14]:
standardized = (selected_x - np.array(base_artifact["mean"])) / np.array(base_artifact["scale"])
contributions = standardized * np.array(base_artifact["coefficients"])
linear_score = float(base_artifact["intercept"] + contributions.sum())
selected_base = float(expit(linear_score))
display(pd.DataFrame({
    "feature": FEATURES,
    "measured value": selected_x,
    "standardized value": standardized,
    "learned beta": base_artifact["coefficients"],
    "contribution to raw score": contributions,
}))
print(f"Intercept: {base_artifact['intercept']:.6f}")
print(f"Sum of feature contributions: {contributions.sum():.6f}")
print(f"Raw score: {linear_score:.6f}")
print(f"Sigmoid(raw score): {selected_base:.6f} = {selected_base:.2%}")
np.testing.assert_allclose(
    selected_base,
    base_model.predict_proba(selected_x.reshape(1, -1))[0, 1],
    atol=1e-12,
)


,feature,measured value,standardized value,learned beta,contribution to raw score
0,log_prompt_chars,7.132498,-0.124024,-0.175905,0.021816
1,log_code_chars,6.643790,0.586114,-0.517597,-0.303370
2,log_code_lines,3.295837,0.598103,-2.091595,-1.250990
3,log_output_tokens,6.919684,0.361739,-0.264206,-0.095574
4,mean_logprob,-0.167347,-0.409071,-0.239882,0.098129
5,p10_logprob,-0.635513,-0.668721,-0.447582,0.299307
6,logprob_sd,0.298148,0.550796,-1.672956,-0.921457
7,low_probability_fraction,0.000000,-0.373777,-0.631655,0.236098
8,ast_depth,10.000000,0.705604,-0.211947,-0.149551
9,branches,2.000000,0.083273,0.048774,0.004062


Intercept: -1.749207
Sum of feature contributions: 1.613414
Raw score: -0.135793
Sigmoid(raw score): 0.466104 = 46.61%


## 9. Fit Platt scaling on separate calibration examples

A classifier can separate stronger and weaker answers yet have poorly scaled probabilities. Platt scaling learns one shared adjustment from the original score to a revised probability.

Here the base model is frozen. A second logistic regression takes only the base logit as input and learns two parameters: a slope and an intercept. It does not re-learn all sixteen feature weights.

The calibration labels are permitted here because this is another fitting step. They are not available when scoring a new answer.


In [15]:
calibration = parts["calibration"]
cal_logits = logit(np.clip(base_scores["calibration"], 1e-6, 1 - 1e-6)).reshape(-1, 1)
platt_model = LogisticRegression(C=1e6, max_iter=2000)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    with threadpool_limits(limits=1):
        platt_model.fit(cal_logits, calibration["y"])

platt_slope = float(platt_model.coef_[0, 0])
platt_intercept = float(platt_model.intercept_[0])
platt_scores = {
    name: expit(platt_slope * logit(np.clip(scores, 1e-6, 1 - 1e-6)) + platt_intercept)
    for name, scores in base_scores.items()
}
selected_platt = float(expit(platt_slope * logit(selected_base) + platt_intercept))
print(f"Learned slope: {platt_slope:.6f}")
print(f"Learned intercept: {platt_intercept:.6f}")
print(f"Attempt 5: {selected_base:.2%} -> {selected_platt:.2%}")
print("A positive slope preserves ranking. These fitted values are close to an identity mapping.")


Learned slope: 0.985614
Learned intercept: 0.052527
Attempt 5: 46.61% -> 47.97%
A positive slope preserves ranking. These fitted values are close to an identity mapping.


## 10. Define the groups and confidence bands

Overall averages can hide subgroup errors. We audit groups such as long code, deeply nested code, and code using imports.

The group definitions are specified by the method. The 75th-percentile thresholds for 'long' and 'nested' come from training features only. Groups can overlap.

Five bands divide current scores into [0,.2), [.2,.4), [.4,.6), [.6,.8), and [.8,1]. A correction applies to a **group AND a current-score band**. It is not a correction based on the average outcome of the entire group regardless of confidence.


In [16]:
display(pd.DataFrame([
    {"group": "all", "definition": "Every candidate"},
    {"group": "long_prompt", "definition": f"log_prompt_chars > {thresholds['long_prompt']:.6f}"},
    {"group": "long_code", "definition": f"log_code_chars > {thresholds['long_code']:.6f}"},
    {"group": "nested_code", "definition": f"ast_depth > {thresholds['nested_code']:.6f}"},
    {"group": "uses_imports", "definition": "At least one import statement"},
    {"group": "empty_code", "definition": "Empty extracted program"},
    {"group": "long_prompt&nested_code", "definition": "Both group conditions"},
]))
initial_audit = pd.DataFrame(audit(
    platt_scores["calibration"], calibration["y"], calibration["groups"],
    calibration["ids"], group_names, protocol["bins"], 40,
))
initial_audit["absolute gap"] = initial_audit["residual"].abs()
display(initial_audit.loc[initial_audit["supported"]].sort_values("absolute gap", ascending=False).head(12))
print("n counts candidates; tasks counts distinct coding problems.")
print("residual = observed pass rate - average predicted probability.")


,group,definition
0,all,Every candidate
1,long_prompt,log_prompt_chars > 7.408379
2,long_code,log_code_chars > 6.917706
3,nested_code,ast_depth > 10.000000
4,uses_imports,At least one import statement
5,empty_code,Empty extracted program
6,long_prompt&nested_code,Both group conditions


,group,bin,n,tasks,mean,rate,residual,supported,absolute gap
12,long_code,2,138,42,0.501358,0.543478,0.042120,True,0.042120
11,long_code,1,167,52,0.300449,0.335329,0.034881,True,0.034881
3,all,3,266,72,0.700441,0.729323,0.028882,True,0.028882
2,all,2,247,78,0.507287,0.493927,-0.013360,True,0.013360
10,long_code,0,142,57,0.106775,0.119718,0.012944,True,0.012944
4,all,4,538,70,0.910757,0.903346,-0.007411,True,0.007411
1,all,1,222,74,0.303000,0.297297,-0.005702,True,0.005702
5,long_prompt,0,338,46,0.028515,0.026627,-0.001888,True,0.001888
25,empty_code,0,608,105,0.001518,0.000000,-0.001518,True,0.001518
0,all,0,837,122,0.021792,0.022700,0.000908,True,0.000908


n counts candidates; tasks counts distinct coding problems.
residual = observed pass rate - average predicted probability.


## 11. Learn and select multicalibration corrections

For each candidate configuration:
1. On calibration data, find the sufficiently supported group-band slice with the largest absolute prediction-versus-outcome gap.
2. Add that slice's residual to its matching scores, clipping to [0,1].
3. Recompute memberships in confidence bands and check again, because scores and overlapping groups change.
4. Stop at tolerance or the 100-update cap.
5. Apply the frozen rules to validation data and measure Brier score.

Brier score is the average squared difference between a probability and its 0/1 label; lower is better. Validation chooses among the six candidate configurations. It does not fit the correction amounts.

Minimum support means distinct tasks, not ten near-duplicate attempts counted as ten independent problems. Empirical stopping on these slices is not a guarantee of population multicalibration.


In [17]:
validation = parts["validation"]
candidate_fits = []
selection_rows = []
for epsilon in protocol["epsilon_grid"]:
    for support in protocol["support_grid"]:
        fitted = fit_patches(
            platt_scores["calibration"], calibration["y"], calibration["groups"],
            calibration["ids"], group_names, protocol["bins"],
            support, epsilon, protocol["max_updates"],
        )
        candidate_scores = replay(platt_scores["validation"], validation["groups"], group_names, fitted)
        validation_brier = float(np.mean((candidate_scores - validation["y"]) ** 2))
        candidate_fits.append((validation_brier, len(fitted["patches"]), fitted))
        selection_rows.append({
            "epsilon": epsilon,
            "minimum distinct tasks": support,
            "updates": len(fitted["patches"]),
            "validation Brier": validation_brier,
            "stopping reason": fitted["status"],
        })
selected_fit = min(candidate_fits, key=lambda candidate: (candidate[0], candidate[1]))[2]
mc_scores = {
    name: replay(platt_scores[name], part["groups"], group_names, selected_fit)
    for name, part in parts.items()
}
display(pd.DataFrame(selection_rows).sort_values(["validation Brier", "updates"]))
print("Selected epsilon:", selected_fit["epsilon"])
print("Selected task support:", selected_fit["support"])
print("Selected update count:", len(selected_fit["patches"]))


,epsilon,minimum distinct tasks,updates,validation Brier,stopping reason
1,0.030000,40,12,0.107600,converged_on_fit_cells
3,0.060000,40,0,0.108455,converged_on_fit_cells
5,0.100000,40,0,0.108455,converged_on_fit_cells
2,0.060000,20,4,0.109036,converged_on_fit_cells
4,0.100000,20,2,0.109210,converged_on_fit_cells
0,0.030000,20,14,0.109734,converged_on_fit_cells


Selected epsilon: 0.03
Selected task support: 40
Selected update count: 12


### Inspect an actual learned correction

This is not an invented '80% becomes 60%' example. The first selected rule below comes from the current fit. The learned correction is observed rate minus average predicted probability.

One offset is added to every matching score; the procedure does not replace every answer with one shared average.


In [18]:
patch_table = pd.DataFrame(selected_fit["patches"])
display(patch_table[["group", "bin", "n", "tasks", "mean", "rate", "delta", "after"]])
first_patch = selected_fit["patches"][0]
print(
    f"First rule: {first_patch['group']}, confidence band "
    f"[{first_patch['bin']/protocol['bins']:.1f}, {(first_patch['bin']+1)/protocol['bins']:.1f})"
)
print(f"{first_patch['n']} candidates from {first_patch['tasks']} distinct tasks")
print(f"Average prediction: {first_patch['mean']:.4%}")
print(f"Observed pass rate: {first_patch['rate']:.4%}")
print(f"Add {first_patch['delta'] * 100:+.4f} percentage points to matching scores.")


,group,bin,n,tasks,mean,rate,delta,after
0,long_code,2,138,42,0.501358,0.543478,0.042120,0.543478
1,long_code,1,167,52,0.300449,0.335329,0.034881,0.335329
2,all,1,193,70,0.315458,0.274611,-0.040847,0.274611
3,long_code,2,134,41,0.497750,0.537313,0.039563,0.537313
4,all,2,218,75,0.516456,0.481651,-0.034805,0.481651
5,long_code,1,132,46,0.280094,0.310606,0.030512,0.310606
6,all,1,191,69,0.309646,0.267016,-0.042630,0.267016
7,long_code,1,120,45,0.275149,0.316667,0.041518,0.316667
8,all,1,167,65,0.307901,0.269461,-0.038440,0.269461
9,long_code,1,120,45,0.278226,0.316667,0.038440,0.316667


First rule: long_code, confidence band [0.4, 0.6)
138 candidates from 42 distinct tasks
Average prediction: 50.1358%
Observed pass rate: 54.3478%
Add +4.2120 percentage points to matching scores.


## 12. Replay the saved rules on the selected answer, without its label

The same fixed rule list is applied in order. Each rule checks the answer's group membership and **current** confidence band. A prior update can move it into a different band.

An answer need not match every rule, or even any rule. No new correction is learned from its test outcome.


In [19]:
selected_groups = membership(selected_x.reshape(1, -1), thresholds, group_names)
current = selected_platt
trace = []
for number, patch in enumerate(selected_fit["patches"], start=1):
    band = min(int(np.clip(current, 0, 1) * selected_fit["bins"]), selected_fit["bins"] - 1)
    matches_group = bool(selected_groups[0, group_names.index(patch["group"])])
    if matches_group and band == patch["bin"]:
        before = current
        current = float(np.clip(current + patch["delta"], 0, 1))
        trace.append({
            "rule": number,
            "group": patch["group"],
            "current band": band,
            "before": before,
            "offset": patch["delta"],
            "after": current,
        })
selected_mc = current
display(pd.DataFrame(trace))
display(pd.DataFrame({
    "stage": ["Base logistic", "Platt", "Multicalibration"],
    "predicted probability of passing": [selected_base, selected_platt, selected_mc],
}))
print("Matching groups:", [name for j, name in enumerate(group_names) if selected_groups[0, j]])
print("Recorded label, revealed for evaluation only:", int(raw["is_correct"][CANDIDATE_INDEX]))
print("One answer cannot establish calibration. Calibration concerns rates across collections of answers.")


,rule,group,current band,before,offset,after
0,5,all,2,0.479683,-0.034805,0.444879


,stage,predicted probability of passing
0,Base logistic,0.466104
1,Platt,0.479683
2,Multicalibration,0.444879


Matching groups: ['all']
Recorded label, revealed for evaluation only: 0
One answer cannot establish calibration. Calibration concerns rates across collections of answers.


## 13. Set the routing policy on validation data, then freeze everything

Confidence estimation and routing are separate decisions. The illustrative policy escalates the lowest-confidence 30% of validation examples.

The cutoff is learned from validation scores and deterministic tie keys, not from test labels. On test data the resulting escalation fraction can differ from 30%.

'Escalate' means select an independently recorded GPT-OSS answer to the same task, paired by candidate index. It is not a live API call, a repair of the failed code, or proof that the alternate is always better. We do not claim actual pricing or latency savings.

The reconstructed fitted objects are saved below before computing the notebook's test metrics. Historical access to this benchmark still means this is not a newly blinded experiment.


In [20]:
all_scores = {"logistic": base_scores, "platt": platt_scores, "multicalibration": mc_scores}
routing_cutoffs = {
    method: budget_threshold(scores["validation"], protocol["validation_escalation_budget"], validation["keys"])
    for method, scores in all_scores.items()
}
reconstruction_frozen = {
    "provenance": {
        "purpose": "New public-data reconstruction; not recovered original fellowship artifacts",
        "authored_date": "2026-10-14",
        "dataset": protocol["dataset"],
        "revision": protocol["revision"],
        "reference_run": REFERENCE_RUN.name,
        "test_previously_inspected": True,
    },
    "models": {
        "logistic": base_artifact,
        "platt": {"coefficient": platt_slope, "intercept": platt_intercept},
        "multicalibration": selected_fit,
    },
    "group_names": group_names,
    "group_thresholds": thresholds,
    "routing_thresholds": routing_cutoffs,
}
(OUTPUT / "reconstructed-model.json").write_text(json.dumps(reconstruction_frozen, indent=2) + "\n")
display(pd.DataFrame([
    {
        "method": method,
        "score cutoff": cutoff["score"],
        "validation escalation": float(route_mask(all_scores[method]["validation"], cutoff, validation["keys"]).mean()),
    }
    for method, cutoff in routing_cutoffs.items()
]))
print("Frozen model saved to reconstructed-model.json")


,method,score cutoff,validation escalation
0,logistic,0.005221,0.300000
1,platt,0.005930,0.300000
2,multicalibration,0.005930,0.300000


Frozen model saved to reconstructed-model.json


## 14. Evaluate the frozen methods on held-out tasks

The labels are now used to score the predictions, not to adjust them. We report probability quality and routing outcomes separately.

A prediction of 0.8 on a passing answer has Brier loss 0.04; on a failing answer, 0.64. Good ranking, good calibration, and good routing are related but not identical objectives.

The cascade can rescue primary failures **and** replace primary successes with failures. Both effects must be counted.


In [21]:
test = parts["test"]
result_rows = []
for method, scores in all_scores.items():
    p = np.clip(scores["test"], 1e-6, 1 - 1e-6)
    route_result = policy(scores["test"], test["y"], test["alternate"], routing_cutoffs[method], test["keys"])
    result_rows.append({
        "method": method,
        "test Brier": float(np.mean((scores["test"] - test["y"]) ** 2)),
        "test log loss": float(-np.mean(test["y"] * np.log(p) + (1 - test["y"]) * np.log(1 - p))),
        "test escalation": route_result["escalation"],
        "routed-policy pass rate": route_result["success"],
        "primary failures rescued": route_result["rescued"],
        "primary successes lost": route_result["regressed"],
    })
results = pd.DataFrame(result_rows)
display(results)
print(f"Always keep primary: {test['y'].mean():.2%} pass rate")
print(f"Always select alternate: {test['alternate'].mean():.2%} pass rate")
print("These are descriptive offline replay outcomes, not measured production performance.")


,method,test Brier,test log loss,test escalation,routed-policy pass rate,primary failures rescued,primary successes lost
0,logistic,0.111126,0.340996,0.247727,0.511742,0.053788,0.000000
1,platt,0.111316,0.341450,0.247727,0.511742,0.053788,0.000000
2,multicalibration,0.112552,0.344726,0.247727,0.511742,0.053788,0.000000


Always keep primary: 45.80% pass rate
Always select alternate: 51.06% pass rate
These are descriptive offline replay outcomes, not measured production performance.


### Does the difference exceed sampling uncertainty?

Resample entire tasks, retaining their ten related attempts together. Resampling individual candidates as if they were independent would overstate the evidence.

The interval below concerns the paired change in Brier score. Positive means the added calibration stage is worse. An interval spanning zero does not establish an improvement or prove that the methods are identical.

These intervals are conditional on the fitted models; they do not include all uncertainty from retraining or choosing the benchmark.


In [22]:
difference_platt = (
    (platt_scores["test"] - test["y"]) ** 2
    - (base_scores["test"] - test["y"]) ** 2
)
difference_mc = (
    (mc_scores["test"] - test["y"]) ** 2
    - (platt_scores["test"] - test["y"]) ** 2
)
interval_rows = []
for name, differences in [
    ("Platt minus logistic", difference_platt),
    ("Multicalibration minus Platt", difference_mc),
]:
    interval = clustered_interval(differences, test["ids"], protocol["seed"], protocol["bootstrap_replicates"])
    interval_rows.append({
        "comparison": name,
        "Brier change": interval["estimate"],
        "95% lower": interval["low"],
        "95% upper": interval["high"],
        "distinct tasks": interval["tasks"],
    })
intervals = pd.DataFrame(interval_rows)
display(intervals)
for item in interval_rows:
    if item["95% lower"] <= 0 <= item["95% upper"]:
        print(item["comparison"] + ": this interval does not establish a clear difference.")
print("Do not rewrite corrections after seeing this table and present the revised result as untouched evaluation.")


,comparison,Brier change,95% lower,95% upper,distinct tasks
0,Platt minus logistic,0.000190,-0.000443,0.000799,264
1,Multicalibration minus Platt,0.001236,-0.000818,0.003303,264


Platt minus logistic: this interval does not establish a clear difference.
Multicalibration minus Platt: this interval does not establish a clear difference.
Do not rewrite corrections after seeing this table and present the revised result as untouched evaluation.


## 15. Put the ten answers back together

The same problem can produce answers with different code, token statistics, confidence estimates, and outcomes. All ten stay in the test partition.

An answer receiving 73% confidence can fail. That is not, by itself, a contradiction: a calibrated group at that score would still contain failures.


In [23]:
positions = [i for i, row in enumerate(test["rows"]) if row["id"] == EXAMPLE_ID]
display(pd.DataFrame([
    {
        "attempt": test["rows"][i]["index"] + 1,
        "code characters": len(test["rows"][i]["code"]),
        "mean token logprob": test["x"][i, FEATURES.index("mean_logprob")],
        "base confidence": base_scores["test"][i],
        "Platt confidence": platt_scores["test"][i],
        "multicalibrated confidence": mc_scores["test"][i],
        "recorded pass/fail": int(test["y"][i]),
    }
    for i in positions
]).sort_values("attempt"))


,attempt,code characters,mean token logprob,base confidence,Platt confidence,multicalibrated confidence,recorded pass/fail
0,1,599,-0.092165,0.548925,0.561199,0.526395,1
1,2,777,-0.027713,0.736369,0.743632,0.743632,1
2,3,777,-0.026949,0.739052,0.746228,0.746228,1
3,4,571,-0.098452,0.609662,0.620578,0.620578,1
4,5,767,-0.167347,0.466104,0.479683,0.444879,0
5,6,755,-0.027054,0.752232,0.758982,0.758982,1
6,7,788,-0.101509,0.590247,0.601630,0.601630,0
7,8,613,-0.049966,0.707712,0.715877,0.715877,1
8,9,798,-0.048990,0.767435,0.773685,0.773685,1
9,10,804,-0.028879,0.721862,0.729586,0.729586,0


## 16. Check that this is a faithful reconstruction of the saved local study

These checks compare the refitted pipeline with the existing recorded local run. They establish computational agreement, not the history of the original fellowship.

All sixteen features for the selected example were already checked against the prepared data. We now check all test probabilities, the task split, and recorded test Brier scores. No historical files are modified.


In [24]:
reference = json.loads((REFERENCE_RUN / "frozen.json").read_text())
reference_summary = json.loads((REFERENCE_RUN / "summary.json").read_text())
reference_splits = json.loads((REFERENCE_RUN / "splits.json").read_text())
assert manifests == reference_splits

reference_base = predict_classifier(reference["models"]["logistic"], test["x"])
reference_platt = expit(
    reference["models"]["platt"]["coefficient"] * logit(np.clip(reference_base, 1e-6, 1 - 1e-6))
    + reference["models"]["platt"]["intercept"]
)
reference_mc = replay(reference_platt, test["groups"], group_names, reference["models"]["multicalibration"])
max_differences = {}
for method, expected in [
    ("logistic", reference_base), ("platt", reference_platt), ("multicalibration", reference_mc),
]:
    observed = all_scores[method]["test"]
    np.testing.assert_allclose(observed, expected, atol=1e-8, rtol=1e-7)
    assert np.isfinite(observed).all()
    assert ((observed >= 0) & (observed <= 1)).all()
    max_differences[method] = float(np.max(np.abs(observed - expected)))
    np.testing.assert_allclose(
        float(np.mean((observed - test["y"]) ** 2)),
        reference_summary["partitions"]["test"]["methods"][method]["brier"],
        atol=1e-8,
    )
display(pd.DataFrame({"method": list(max_differences), "maximum prediction difference": list(max_differences.values())}))
print("PASS: split integrity, finite bounded probabilities, refit agreement, and recorded metrics.")

report = {
    "scope": "Public-data reconstruction, not recovered original work",
    "features": FEATURES,
    "partitions": {name: {"tasks": len(manifests[name]), "candidates": len(part["y"])} for name, part in parts.items()},
    "results": result_rows,
    "paired_intervals": interval_rows,
    "maximum_prediction_difference_from_reference": max_differences,
    "example": {"task": EXAMPLE_ID, "attempt": CANDIDATE_INDEX + 1, "features": selected_x.tolist(),
                "logistic": selected_base, "platt": selected_platt, "multicalibration": selected_mc,
                "recorded_label": int(raw["is_correct"][CANDIDATE_INDEX])},
    "checks_passed": True,
}
(OUTPUT / "reconstruction-results.json").write_text(json.dumps(report, indent=2, allow_nan=False) + "\n")
print("Results saved to reconstruction-results.json")


,method,maximum prediction difference
0,logistic,0.000000
1,platt,0.000000
2,multicalibration,0.000000


PASS: split integrity, finite bounded probabilities, refit agreement, and recorded metrics.
Results saved to reconstruction-results.json


## 17. What this project is, in plain English

**Machine learning:** fitting the logistic confidence predictor from labeled examples. Platt scaling and the conditional corrections also learn adjustments from data.

**Data analysis:** inspecting the raw records, missing/empty answers, feature distributions, and subgroup errors.

**Evaluation:** checking probability quality and routing decisions on tasks that did not fit the predictor or select its settings.

**Engineering:** implementing the extraction, partitioning, fitting, serialized rules, and reproducible evaluation pipeline.

**Not LLM training:** neither the primary nor alternate coding model is retrained here.

### A natural description of the reconstruction
> I reconstructed a confidence-estimation experiment for AI-generated code. I used public coding problems, recorded model outputs, and published pass/fail labels. I converted each answer into numerical features, fitted a logistic model to estimate its probability of passing, and tested whether score-only and subgroup calibration improved those estimates. I kept related answers together when splitting the data and compared the methods on separate tasks.

Use historical 'I did' wording only for parts you actually performed originally. This new notebook shows how the workflow works; it cannot recover the original dataset, editor, exact feature list, or historical results.

### What the result teaches
The extra calibration stages did not establish a clear test Brier improvement in this reconstruction. A technically reasonable idea still needs empirical testing. Reporting that outcome is more accurate than inventing a win.

### Important limitations
- One benchmark, a fixed partition, and already inspected test results.
- Publisher labels and extracted programs were not independently re-evaluated.
- Empty extracted programs can reflect extraction failures and materially affect results.
- Token statistics summarize full responses, not code-only tokens.
- Group-and-band calibration is empirical; no universal correctness or population guarantee.
- No private data, live routing, actual bills, measured latency, or deployment claims.
- The fixed budget policy is an illustrative rule, not an optimal economic decision.
